# Getting started with TinyTimeMixer (TTM)

This notebooke demonstrates the usage of a pre-trained `TinyTimeMixer` model for several multivariate time series forecasting tasks. For details related to model architecture, refer to the [TTM paper](https://arxiv.org/pdf/2401.03955.pdf).

In this example, we will use a pre-trained TTM-512-96 model. That means the TTM model can take an input of 512 time points (`context_length`), and can forecast upto 96 time points (`forecast_length`) in the future. We will use the pre-trained TTM in two settings:
1. **Zero-shot**: The pre-trained TTM will be directly used to evaluate on the `test` split of the target data. Note that the TTM was NOT pre-trained on the target data.
2. **Few-shot**: The pre-trained TTM will be quickly fine-tuned on only 5% of the `train` split of the target data, and subsequently, evaluated on the `test` part of the target data.

Note: Alternatively, this notebook can be modified to try any other TTM model from a suite of TTM models. For details, visit the [Hugging Face TTM Model Repository](https://huggingface.co/ibm-granite/granite-timeseries-ttm-r2).

1. IBM Granite TTM-R1 pre-trained models can be found here: [Granite-TTM-R1 Model Card](https://huggingface.co/ibm-granite/granite-timeseries-ttm-r1)
2. IBM Granite TTM-R2 pre-trained models can be found here: [Granite-TTM-R2 Model Card](https://huggingface.co/ibm-granite/granite-timeseries-ttm-r2)
3. Research-use (non-commercial use only) TTM-R2 pre-trained models can be found here: [Research-Use-TTM-R2](https://huggingface.co/ibm-research/ttm-research-r2)

### The get_model() utility
TTM Model card offers a suite of models with varying `context_length` and `prediction_length` combinations.
In this notebook, we will utilize the TSFM `get_model()` utility that automatically selects the right model based on the given input `context_length` and `prediction_length` (and some other optional arguments) abstracting away the internal complexity. See the usage examples below in the `zeroshot_eval()` and `fewshot_finetune_eval()` functions. For more details see the [docstring](https://github.com/ibm-granite/granite-tsfm/blob/main/tsfm_public/toolkit/get_model.py) of the function definition.

## Install `tsfm` 
**[Optional for Local Run / Mandatory for Google Colab]**  
Run the below cell to install `tsfm`. Skip if already installed.

In [9]:
# # Install the tsfm library
# ! pip install "granite-tsfm[notebooks] @ git+https://github.com/ibm-granite/granite-tsfm.git@v0.3.3"

## Imports

In [ ]:
import math
import os
import tempfile

import pandas as pd
import numpy as np
from torch.optim import AdamW
from torch.optim.lr_scheduler import OneCycleLR
from transformers import EarlyStoppingCallback, Trainer, TrainingArguments, set_seed
from transformers.integrations import INTEGRATION_TO_CALLBACK

from tsfm_public import TimeSeriesPreprocessor, TrackingCallback, count_parameters, get_datasets
from tsfm_public.toolkit.get_model import get_model
from tsfm_public.toolkit.lr_finder import optimal_lr_finder
from tsfm_public.toolkit.visualization import plot_predictions
import warnings
from tsfm_public.models.tinytimemixer.configuration_tinytimemixer import TinyTimeMixerConfig
from tsfm_public.models.tinytimemixer.modeling_tinytimemixer import TinyTimeMixerForPrediction

# Suppress all warnings
warnings.filterwarnings("ignore")

## Zero-shot evaluation method

In [ ]:
def zeroshot_eval(dataset_name, batch_size, data, context_length=512, forecast_length=96, ):
    # Get data

    tsp = TimeSeriesPreprocessor(
        **column_specifiers,
        context_length=context_length,
        prediction_length=forecast_length,
        scaling=True,
        encode_categorical=False,
        scaler_type="standard",
    )

    config = TinyTimeMixerConfig(context_length=context_length, prediction_length=forecast_length)

    # Load model
    zeroshot_model =  TinyTimeMixerForPrediction.from_pretrained(
    TTM_MODEL_PATH,
    config=config,
    ignore_mismatched_sizes=True
)


    # print(f"Model loss: {zeroshot_model.loss}")
    # print(f"Model config: {zeroshot_model.config}")
    dset_train, dset_valid, dset_test = get_datasets(
        tsp, data, split_config, use_frequency_token=zeroshot_model.config.resolution_prefix_tuning
    )
    # print(dset_test)
    temp_dir = tempfile.mkdtemp()
    # zeroshot_trainer
    zeroshot_trainer = Trainer(
        model=zeroshot_model,
        args=TrainingArguments(
            output_dir=temp_dir,
            per_device_eval_batch_size=batch_size,
            seed=SEED,
            report_to="none",
        ),
    )
    # evaluate = zero-shot performance
    # print("+" * 20, "Test MSE zero-shot", "+" * 20)
    zeroshot_output = zeroshot_trainer.evaluate(dset_test)
    print(zeroshot_output)

    # get predictions

    predictions_dict = zeroshot_trainer.predict(dset_test)
    # print(zeroshot_trainer.model.loss)
    predictions_np = predictions_dict.predictions[0]
    # print(len(predictions_dict))
    # print(predictions_np.shape)
    # print(predictions_np)
    # get backbone embeddings (if needed for further analysis)

    backbone_embedding = predictions_dict.predictions[1]

    # print(backbone_embedding.shape)

    # plot
    # plot_predictions(
    #     model=zeroshot_trainer.model,
    #     dset=dset_test,
    #     plot_dir=os.path.join(OUT_DIR, dataset_name),
    #     plot_prefix="test_zeroshot",
    #     indices=[685, 118, 902, 1984, 894, 967, 304, 57, 265, 1015],
    #     channel=0,
    #     plot_context=context_length
    # )
    return dset_test, predictions_np

# Zeroshot

In [12]:
# dset_test, preds=zeroshot_eval(
#     dataset_name=TARGET_DATASET, context_length=CONTEXT_LENGTH, forecast_length=PREDICTION_LENGTH, batch_size=64
# )

In [13]:
import numpy as np
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

def calculate_metrics(dset_test, preds, model_name="Model",):
    """
    Calculate comprehensive evaluation metrics.
    
    Args:
        dset_test: Test dataset
        preds: Predictions array
        model_name: Name to display in output
    
    Returns:
        dict: Dictionary containing all metrics
    
    Notes:
        MASE: Scaled against the naive 1-step forecast error (mean |y_t - y_{t-1}|)
              computed from the context window (past_values).
        CRPS: For a deterministic/point forecast, CRPS == MAE. Install
              `properscoring` and pass forecast samples/distributions for a 
              proper probabilistic CRPS.
    """
    # Extract ground truth and past values from dset_test
    y_true_list = []
    past_list = []
    for i in range(len(dset_test)):
        sample = dset_test[i]
        
        future_values = sample['future_values']
        if hasattr(future_values, "detach"):
            future_values = future_values.detach().cpu().numpy()
        else:
            future_values = np.asarray(future_values)
        y_true_list.append(future_values)
        
        past_values = sample['past_values']
        if hasattr(past_values, "detach"):
            past_values = past_values.detach().cpu().numpy()
        else:
            past_values = np.asarray(past_values)
        past_list.append(past_values)
    
    y_true = np.array(y_true_list)   # (samples, forecast_len, channels)
    past_arr = np.array(past_list)   # (samples, context_len, channels)
    
    # Validate shapes match
    if preds.shape != y_true.shape:
        raise ValueError(f"Shape mismatch! Predictions: {preds.shape}, Ground truth: {y_true.shape}")
    
    # Flatten for overall metric calculations
    y_true_flat = y_true.flatten()
    preds_flat = preds.flatten()
    epsilon = 1e-8
    
    # Overall metrics
    mse   = float(mean_squared_error(y_true_flat, preds_flat))
    rmse  = float(np.sqrt(mse))
    mae   = float(mean_absolute_error(y_true_flat, preds_flat))
    mape  = float(np.mean(np.abs((y_true_flat - preds_flat) / (y_true_flat + epsilon))) * 100)
    r2    = float(r2_score(y_true_flat, preds_flat))
    smape = float(np.mean(2.0 * np.abs(preds_flat - y_true_flat) / (np.abs(preds_flat) + np.abs(y_true_flat) + epsilon)) * 100)
    
    # MASE: naive 1-step scale from context window
    # naive_scale = mean |y_t - y_{t-1}| over all samples and channels
    naive_scale_overall = float(np.mean(np.abs(np.diff(past_arr, axis=1))) + epsilon)
    mase = mae / naive_scale_overall
    
    # CRPS (deterministic): equals MAE for point forecasts
    crps = mae
    
    # Per-channel metrics
    num_samples, num_timesteps, num_channels = y_true.shape
    
    per_channel_metrics = {}
    for channel in range(num_channels):
        channel_name = target_columns[channel] if channel < len(target_columns) else f"Channel {channel}"
        
        y_true_channel = y_true[:, :, channel].flatten()
        preds_channel  = preds[:, :, channel].flatten()
        
        ch_mse   = float(mean_squared_error(y_true_channel, preds_channel))
        ch_rmse  = float(np.sqrt(ch_mse))
        ch_mae   = float(mean_absolute_error(y_true_channel, preds_channel))
        ch_mape  = float(np.mean(np.abs((y_true_channel - preds_channel) / (y_true_channel + epsilon))) * 100)
        ch_r2    = float(r2_score(y_true_channel, preds_channel))
        ch_smape = float(np.mean(2.0 * np.abs(preds_channel - y_true_channel) / (np.abs(preds_channel) + np.abs(y_true_channel) + epsilon)) * 100)
        
        # Per-channel MASE: scale from that channel's context differences
        ch_naive_scale = float(np.mean(np.abs(np.diff(past_arr[:, :, channel], axis=1))) + epsilon)
        ch_mase = ch_mae / ch_naive_scale
        
        # CRPS (point forecast) = MAE per channel
        ch_crps = ch_mae
        
        per_channel_metrics[channel_name] = {
            'mse':   ch_mse,
            'rmse':  ch_rmse,
            'mae':   ch_mae,
            'mape':  ch_mape,
            'r2':    ch_r2,
            'smape': ch_smape,
            'mase':  ch_mase,
            'crps':  ch_crps,
        }
    
    return {
        'overall': {
            'mse':   mse,
            'rmse':  rmse,
            'mae':   mae,
            'mape':  mape,
            'r2':    r2,
            'smape': smape,
            'mase':  mase,
            'crps':  crps,
        },
        'per_channel': per_channel_metrics
    }

# Calculate metrics for TTM predictions
# ttm_metrics = calculate_metrics(dset_test, preds, model_name="TTM Zero-Shot")


In [14]:
def calculate_baseline_metrics(dset, method='mean'):
    """
    Calculate baseline metrics using simple statistical methods.
    
    Args:
        dset: Dataset containing past_values and future_values
        method: 'mean' or 'median' - aggregation method for baseline
    
    Returns:
        tuple: (predictions array, metrics dictionary)
    """
    preds = []
    y_true_list = []
    past_list = []
    
    for i in range(len(dset)):
        past_values = dset[i]['past_values']
        if hasattr(past_values, "detach"):
            past = past_values.detach().cpu().numpy()
        else:
            past = np.asarray(past_values)
        past_list.append(past)
        
        if method == 'mean':
            baseline_value = np.mean(past, axis=0)
        elif method == 'median':
            baseline_value = np.median(past, axis=0)
        else:
            raise ValueError(f"Unknown method: {method}. Use 'mean' or 'median'")
        
        pred = np.tile(baseline_value, (PREDICTION_LENGTH, 1))
        preds.append(pred)

        future_values = dset[i]['future_values']
        if hasattr(future_values, "detach"):
            future_values = future_values.detach().cpu().numpy()
        else:
            future_values = np.asarray(future_values)
        y_true_list.append(future_values)
    
    y_true = np.array(y_true_list)
    preds = np.array(preds)
    past_arr = np.array(past_list)   # (samples, context_len, channels)
    
    # Validate shapes match
    if preds.shape != y_true.shape:
        raise ValueError(f"Shape mismatch! Predictions: {preds.shape}, Ground truth: {y_true.shape}")
    
    # Calculate overall metrics
    y_true_flat = y_true.flatten()
    preds_flat = preds.flatten()
    epsilon = 1e-8
    
    mse = float(mean_squared_error(y_true_flat, preds_flat))
    rmse = float(np.sqrt(mse))
    mae = float(mean_absolute_error(y_true_flat, preds_flat))
    mape = float(np.mean(np.abs((y_true_flat - preds_flat) / (y_true_flat + epsilon))) * 100)
    r2 = float(r2_score(y_true_flat, preds_flat))
    smape = float(np.mean(2.0 * np.abs(preds_flat - y_true_flat) / (np.abs(preds_flat) + np.abs(y_true_flat) + epsilon)) * 100)

    # Overall MASE and CRPS (same approach as TTM metrics)
    naive_scale_overall = float(np.mean(np.abs(np.diff(past_arr, axis=1))) + epsilon)
    mase = mae / naive_scale_overall
    crps = mae
    
    # Calculate per-channel metrics
    num_samples, num_timesteps, num_channels = y_true.shape
    
    per_channel_metrics = {}
    for channel in range(num_channels):
        channel_name = target_columns[channel] if channel < len(target_columns) else f"Channel {channel}"
        
        # Extract channel data
        y_true_channel = y_true[:, :, channel].flatten()
        preds_channel = preds[:, :, channel].flatten()
        
        # Calculate metrics for this channel
        ch_mse = float(mean_squared_error(y_true_channel, preds_channel))
        ch_rmse = float(np.sqrt(ch_mse))
        ch_mae = float(mean_absolute_error(y_true_channel, preds_channel))
        ch_mape = float(np.mean(np.abs((y_true_channel - preds_channel) / (y_true_channel + epsilon))) * 100)
        ch_r2 = float(r2_score(y_true_channel, preds_channel))
        ch_smape = float(np.mean(2.0 * np.abs(preds_channel - y_true_channel) / (np.abs(preds_channel) + np.abs(y_true_channel) + epsilon)) * 100)
        
        # Per-channel MASE: scale from that channel's context differences
        ch_naive_scale = float(np.mean(np.abs(np.diff(past_arr[:, :, channel], axis=1))) + epsilon)
        ch_mase = ch_mae / ch_naive_scale
        
        # CRPS (point forecast) = MAE per channel
        ch_crps = ch_mae
        
        per_channel_metrics[channel_name] = {
            'mse': ch_mse,
            'rmse': ch_rmse,
            'mae': ch_mae,
            'mape': ch_mape,
            'r2': ch_r2,
            'smape': ch_smape,
            'mase': ch_mase,
            'crps': ch_crps,
        }
    
    metrics = {
        'overall': {
            'mse': mse,
            'rmse': rmse,
            'mae': mae,
            'mape': mape,
            'r2': r2,
            'smape': smape,
            'mase': mase,
            'crps': crps,
        },
        'per_channel': per_channel_metrics
    }
    
    return preds, metrics


# Calculate baseline metrics
# mean_baseline_preds, mean_baseline_metrics = calculate_baseline_metrics(dset_test, method='mean')
# median_baseline_preds, median_baseline_metrics = calculate_baseline_metrics(dset_test, method='median')


In [15]:
SEED = 42
set_seed(SEED)

# TTM Model path. The default model path is Granite-R2. Below, you can choose other TTM releases.
TTM_MODEL_PATH = "ibm-granite/granite-timeseries-ttm-r2"
# TTM_MODEL_PATH = "ibm-granite/granite-timeseries-ttm-r1"
# TTM_MODEL_PATH = "ibm-research/ttm-research-r2"

# Context length, Or Length of the history.
# Currently supported values are: 512/1024/1536 for Granite-TTM-R2 and Research-Use-TTM-R2, and 512/1024 for Granite-TTM-R1
CONTEXT_LENGTH = 512
#1  week or 2 weeks, predict for next 2 days
# Granite-TTM-R2 supports forecast length upto 720 and Granite-TTM-R1 supports forecast length upto 96
# Arima? Rolling average, Rolling median 
PREDICTION_LENGTH = 96
OUT_DIR = "ttm_finetuned_models/"

In [16]:
import json
from datetime import datetime
from tqdm import tqdm

folder = r"/home/rishi/ML Projects/Air Pollution/CPCB/sites_imputed"
files = os.listdir(folder)  # Fixed - get all files in the folder
timestamp_column = "Timestamp"
id_columns = []  # mention the ids that uniquely identify a time-series.

target_columns = [
 'PM2.5 (µg/m³)',
 'PM10 (µg/m³)',
 'NO2 (µg/m³)',
 'SO2 (µg/m³)',
 'CO (mg/m³)',
 'Ozone (µg/m³)',
]

split_config = {
    "train": 0.6,
    "test": 0.2,
}

column_specifiers = {
    "timestamp_column": timestamp_column,
    "id_columns": id_columns,
    "target_columns": target_columns,
    "control_columns": [],
}

# Create output directory for results
results_dir = "ttm_benchmarking_results_mase_v1"
os.makedirs(results_dir, exist_ok=True)

# Store all results
all_results = []

for file in tqdm(files):
    if not file.endswith('.csv'):
        continue
        
    print(f"\n{'='*60}")
    print(f"Processing: {file}")
    print(f"{'='*60}")
    
    try:
        data = pd.read_csv(
            os.path.join(folder, file),
            parse_dates=[timestamp_column],
        ).copy()
        
        site_name = file.replace('.csv', '')
        
        # Run zero-shot evaluation
        dset_test, preds = zeroshot_eval(
            dataset_name=site_name, 
            data=data,
            context_length=CONTEXT_LENGTH, 
            forecast_length=PREDICTION_LENGTH, 
            batch_size=64
        )
        
        # Calculate TTM metrics
        ttm_metrics = calculate_metrics(dset_test, preds, model_name=f"TTM - {site_name}")
        
        # Calculate baseline metrics (mean)
        mean_baseline_preds, mean_baseline_metrics = calculate_baseline_metrics(dset=dset_test, method='mean')
        
        # Calculate baseline metrics (median)
        median_baseline_preds, median_baseline_metrics = calculate_baseline_metrics(dset=dset_test, method='median')
        
        # Store results
        result = {
            'site': site_name,
            'file': file,
            'timestamp': datetime.now().isoformat(),
            'context_length': CONTEXT_LENGTH,
            'prediction_length': PREDICTION_LENGTH,
            'ttm_metrics': ttm_metrics,
            'mean_baseline_metrics': mean_baseline_metrics,
            'median_baseline_metrics': median_baseline_metrics
        }
        all_results.append(result)
        
        # Save individual site results
        site_result_file = os.path.join(results_dir, f"{site_name}_metrics.json")
        with open(site_result_file, 'w') as f:
            json.dump(result, f, indent=2)
        print(f"Saved results to: {site_result_file}")
        
    except Exception as e:
        print(f"Error processing {file}: {str(e)}")
        continue

# Save combined results
combined_results_file = os.path.join(results_dir, f"all_sites_metrics_{datetime.now().strftime('%Y%m%d_%H%M%S')}.json")
with open(combined_results_file, 'w') as f:
    json.dump(all_results, f, indent=2)
print(f"\n{'='*60}")
print(f"All results saved to: {combined_results_file}")
print(f"{'='*60}")

  0%|          | 0/138 [00:00<?, ?it/s]INFO:p-995304:t-125149693071488:get_model.py:get_model:Loading model from: ibm-granite/granite-timeseries-ttm-r2



Processing: site_1431_Patparganj_Delhi_DPCC_15Min.csv


INFO:p-995304:t-125149693071488:get_model.py:get_model:Model loaded successfully from ibm-granite/granite-timeseries-ttm-r2, revision = main.
INFO:p-995304:t-125149693071488:get_model.py:get_model:[TTM] context_length = 512, prediction_length = 96


{'eval_loss': 0.4748593866825104, 'eval_model_preparation_time': 0.0007, 'eval_runtime': 1.2158, 'eval_samples_per_second': 4248.22, 'eval_steps_per_second': 66.623}


  1%|          | 1/138 [00:06<15:15,  6.68s/it]INFO:p-995304:t-125149693071488:get_model.py:get_model:Loading model from: ibm-granite/granite-timeseries-ttm-r2


...

Saved results to: ttm_benchmarking_results_mase_v1/site_5541_Mayaganj_Bhagalpur_BSPCB_15Min_metrics.json

Processing: site_1560_Bawana_Delhi_DPCC_15Min.csv


INFO:p-995304:t-125149693071488:get_model.py:get_model:Model loaded successfully from ibm-granite/granite-timeseries-ttm-r2, revision = main.
INFO:p-995304:t-125149693071488:get_model.py:get_model:[TTM] context_length = 512, prediction_length = 96


{'eval_loss': 0.5092381238937378, 'eval_model_preparation_time': 0.0007, 'eval_runtime': 3.2844, 'eval_samples_per_second': 1572.568, 'eval_steps_per_second': 24.662}


100%|██████████| 138/138 [21:00<00:00,  9.14s/it]

Saved results to: ttm_benchmarking_results_mase_v1/site_1560_Bawana_Delhi_DPCC_15Min_metrics.json

All results saved to: ttm_benchmarking_results_mase_v1/all_sites_metrics_20260224_134821.json
